# U-Net Glomeruli Segmentation — Binary Segmentation Training Pipeline

This notebook implements a complete PyTorch U-Net training system for binary semantic segmentation of glomeruli from renal biopsy whole-slide images.

## Two-Model Strategy

This pipeline uses a **two-stage approach** for glomeruli classification:

1. **Model 1 (this notebook)**: Binary segmentation to detect glomerulus locations
   - Input: Z-score normalized RGB tiles from WSI
   - Output: Pixel-level binary masks (background vs glomerulus)
   - Classes: 0 (background), 1 (glomerulus — fused from original classes 1-4)
   - Purpose: Localizes all glomeruli regardless of subtype

2. **Model 2 (future)**: Classifier on detected patches to assign classes 1-4
   - Input: Image patches extracted from detected glomerulus regions (Model 1)
   - Output: Per-glomerulus classification (classes 1-4: FGS, MPGN, FSGS, other)
   - Purpose: Fine-grained subtype classification for clinical diagnosis

## Notebook Contents

1. **Data Loading** — GlomeruliDataset with grouped split and online augmentation (train only)
2. **Loss Functions** — BCEWithLogits + Dice for binary segmentation
3. **U-Net Architecture** — Encoder-Decoder with Skip Connections
4. **Binary Segmentation Metrics** — Accuracy, Precision, Recall, F1, ROC-AUC, Dice
5. **Training Pipeline** — Full training loop with validation and checkpointing

## Pipeline Overview

WSI TIFF + GeoJSON annotations → Tiled images + masks (classes 0-4) →
Reinhard color normalization → Z-score standardization →
Train/Val/Test split (grouped by biopsy, 70/15/15) →
Online augmentation (train only) → U-Net training → Model evaluation (Binary Metrics)

See WORKFLOW.md for end-to-end pipeline documentation.

In [ ]:
# Standard library
import os
import json
import time
from pathlib import Path
from datetime import datetime
from typing import Tuple
import random

# Scientific computing
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR
from torch.cuda.amp import autocast, GradScaler
from torch.utils.tensorboard import SummaryWriter
import albumentations as A

# Data loading
import cv2
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, WeightedRandomSampler
import warnings

# Progress/model library imports
from tqdm.auto import tqdm
import segmentation_models_pytorch as smp

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# --- CUDA performance tuning (T4 Tensor Cores) ---
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True          # auto-tune conv algorithms
    torch.backends.cuda.matmul.allow_tf32 = True   # TF32 for matmul (T4 supports it)
    torch.backends.cudnn.allow_tf32 = True         # TF32 for cudnn convolutions
    print(f"cudnn.benchmark: {torch.backends.cudnn.benchmark}")
    print(f"CUDA matmul TF32: {torch.backends.cuda.matmul.allow_tf32}")
    print(f"cudnn TF32: {torch.backends.cudnn.allow_tf32}")
    print(f"PYTORCH_CUDA_ALLOC_CONF: expandable_segments:True")


# --- Preprocessing Transforms (Reinhard Normalization + Z-score) ---

class ReinhardNormalize:
    """Reinhard stain normalization in LAB color space for consistency across slides."""

    def __init__(self, target_stats: dict):
        self.target = target_stats

    @staticmethod
    def _get_tissue_mask(img_bgr: np.ndarray) -> np.ndarray:
        """Isolate tissue pixels from background and artifacts via luminance and saturation thresholds."""
        lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
        hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
        mask = (lab[:, :, 0] < 230) & (hsv[:, :, 1] > 10)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
        mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE, kernel)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        return mask.astype(bool)

    @staticmethod
    def compute_template_stats(image_paths: list, n_samples: int = 200) -> dict:
        """Compute median LAB statistics from tissue pixels to define normalization target."""
        paths_list = list(image_paths)[:n_samples]
        sample = random.sample(paths_list, min(n_samples, len(paths_list)))
        all_stats = []
        
        for p in sample:
            img = cv2.imread(str(p))
            if img is None:
                continue
            tissue = ReinhardNormalize._get_tissue_mask(img)
            if tissue.sum() < 100:
                continue
            lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB).astype(np.float32)
            stats = ([lab[..., c][tissue].mean() for c in range(3)] +
                     [lab[..., c][tissue].std()  for c in range(3)])
            all_stats.append(stats)
        
        if not all_stats:
            return {'mean_L': 50, 'mean_a': 128, 'mean_b': 128,
                    'std_L': 10, 'std_a': 10, 'std_b': 10}
        
        arr = np.array(all_stats)
        keys = ['mean_L', 'mean_a', 'mean_b', 'std_L', 'std_a', 'std_b']
        return {k: float(np.median(arr[:, i])) for i, k in enumerate(keys)}

    def __call__(self, img_bgr: np.ndarray) -> np.ndarray:
        """Normalize image to match template statistics, preserving background pixels."""
        tissue = self._get_tissue_mask(img_bgr)
        if tissue.sum() < 100:
            return img_bgr
        
        lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
        
        src = {c: (lab[..., i][tissue].mean(), lab[..., i][tissue].std())
               for i, c in enumerate(['L', 'a', 'b'])}
        
        result = lab.copy()
        for i, c in enumerate(['L', 'a', 'b']):
            m, s = src[c]
            result[..., i] = ((lab[..., i] - m) *
                              (self.target[f'std_{c}'] / (s + 1e-5)) +
                              self.target[f'mean_{c}'])
        
        result[~tissue] = lab[~tissue]
        result = np.clip(result, 0, 255).astype(np.uint8)
        return cv2.cvtColor(result, cv2.COLOR_LAB2BGR)


def compute_channel_stats(image_paths: list, n_samples: int = 200) -> Tuple[list, list]:
    """Compute weighted per-channel RGB stats from tissue pixels for Z-score normalization."""
    paths_list = list(image_paths)[:n_samples]
    sample = random.sample(paths_list, min(n_samples, len(paths_list)))
    
    tile_means, tile_vars, tile_counts = [], [], []
    for p in sample:
        img_bgr = cv2.imread(str(p))
        if img_bgr is None:
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        tissue = ReinhardNormalize._get_tissue_mask(img_bgr)
        if tissue.sum() < 100:
            continue
        
        pixels = img_rgb[tissue]
        tile_means.append(pixels.mean(0))
        tile_vars.append(pixels.var(0))
        tile_counts.append(tissue.sum())
    
    if not tile_means:
        return [0.5, 0.5, 0.5], [0.2, 0.2, 0.2]
    
    means_arr = np.array(tile_means)
    vars_arr = np.array(tile_vars)
    counts_arr = np.array(tile_counts, dtype=np.float64)
    w = counts_arr / counts_arr.sum()
    
    mean = (means_arr * w[:, None]).sum(0)
    var = ((vars_arr + (means_arr - mean) ** 2) * w[:, None]).sum(0)
    
    return mean.tolist(), np.sqrt(var).tolist()

In [ ]:
def _collect_image_tiles(images_dir: str) -> list:
    """Collect only input tiles from */images/*.png, never generated masks."""
    images_dir = Path(images_dir)
    image_paths = sorted(images_dir.glob('*/images/*.png'))

    # Fallback for flat/custom datasets: include PNGs except anything inside a masks folder
    # or files already named *_mask.png.
    if not image_paths:
        image_paths = sorted(
            p for p in images_dir.rglob('*.png')
            if 'masks' not in p.relative_to(images_dir).parts
            and not p.stem.endswith('_mask')
        )

    return image_paths


def _slide_name_from_image_path(image_path, images_dir) -> str:
    """Infer the biopsy/slide name from a tile path under <root>/<slide>/images/*.png."""
    image_path = Path(image_path)
    images_dir = Path(images_dir)
    try:
        rel = image_path.relative_to(images_dir)
        if rel.parts:
            return rel.parts[0]
    except ValueError:
        pass

    if image_path.parent.name == 'images' and image_path.parent.parent.name:
        return image_path.parent.parent.name
    return image_path.parent.name or image_path.stem


def split_biopsias(
    images_dir: str,
    train_size: float = 0.70,
    val_size: float = 0.15,
    seed: int = 42,
) -> Tuple[list, list, list, dict]:
    """Groups biopsias into train/val/test to prevent data leakage at slide level."""
    images_dir = Path(images_dir)

    all_images = _collect_image_tiles(images_dir)

    if not all_images:
        raise ValueError(f"No PNG image tiles found in {images_dir}. Expected files under */images/*.png")

    biopsias_dict = {}
    for img_path in all_images:
        relative = img_path.relative_to(images_dir)
        biopsia = relative.parts[0]
        if biopsia not in biopsias_dict:
            biopsias_dict[biopsia] = []
        biopsias_dict[biopsia].append(img_path)

    biopsias_list = list(biopsias_dict.keys())

    test_size = 1.0 - train_size - val_size
    assert test_size >= 0, "train_size + val_size must be <= 1.0"

    if test_size > 0:
        train_val_biopsias, test_biopsias = train_test_split(
            biopsias_list,
            test_size=test_size,
            random_state=seed,
        )
    else:
        train_val_biopsias = biopsias_list
        test_biopsias = []

    if val_size > 0:
        val_fraction = val_size / (train_size + val_size)
        train_biopsias, val_biopsias = train_test_split(
            train_val_biopsias,
            test_size=val_fraction,
            random_state=seed + 1,
        )
    else:
        train_biopsias = train_val_biopsias
        val_biopsias = []

    return train_biopsias, val_biopsias, test_biopsias, biopsias_dict


# ============================================================================
# Grayscale pixel values that correspond to glomerulus classes (any > 0 in practice)
# 64=No_Proliferativo, 128=Proliferativo, 192=Esclerosado, 255=Excluido/Excluyente
# Excluido (255) is INTENTIONALLY mapped to Glomerulus class 1 for binary segmentation
# ============================================================================
class GlomeruliDataset(Dataset):
    """Loads paired image-mask glomeruli tiles with online preprocessing and augmentation."""

    def __init__(
        self,
        images_dir: str,
        masks_dir: str = None,
        split: str = 'train',
        biopsias: list = None,
        biopsias_dict: dict = None,
        reinhard_norm=None,
        channel_means: list = None,
        channel_stds: list = None,
        train_size: float = 0.70,
        val_size: float = 0.15,
        seed: int = 42,
        transforms=None,
    ):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir) if masks_dir is not None else Path(images_dir)
        self.split = split
        self.transforms = transforms
        self.reinhard_norm = reinhard_norm
        self.channel_means = channel_means if channel_means is not None else [0.5, 0.5, 0.5]
        self.channel_stds = channel_stds if channel_stds is not None else [0.2, 0.2, 0.2]

        assert split in {'train', 'val', 'test'}, f"Invalid split: {split}"
        assert self.images_dir.exists(), f"Images dir not found: {self.images_dir}"
        assert self.masks_dir.exists(), f"Masks dir not found: {self.masks_dir}"

        if biopsias is not None and biopsias_dict is not None:
            selected_biopsias = biopsias
            self.biopsias_dict = biopsias_dict
        else:
            test_size = 1.0 - train_size - val_size
            assert test_size >= 0, "train_size + val_size must be <= 1.0"

            all_images = _collect_image_tiles(self.images_dir)
            if not all_images:
                raise ValueError(f"No PNG image tiles found in {self.images_dir}. Expected files under */images/*.png")

            biopsias_dict = {}
            for img_path in all_images:
                relative = img_path.relative_to(self.images_dir)
                biopsia = relative.parts[0]
                if biopsia not in biopsias_dict:
                    biopsias_dict[biopsia] = []
                biopsias_dict[biopsia].append(img_path)

            self.biopsias_dict = biopsias_dict
            biopsias_list = list(biopsias_dict.keys())

            if test_size > 0:
                train_val_biopsias, test_biopsias = train_test_split(
                    biopsias_list,
                    test_size=test_size,
                    random_state=seed,
                )
            else:
                train_val_biopsias = biopsias_list
                test_biopsias = []

            if val_size > 0:
                val_fraction = val_size / (train_size + val_size)
                train_biopsias, val_biopsias = train_test_split(
                    train_val_biopsias,
                    test_size=val_fraction,
                    random_state=seed + 1,
                )
            else:
                train_biopsias = train_val_biopsias
                val_biopsias = []

            if split == 'train':
                selected_biopsias = train_biopsias
            elif split == 'val':
                selected_biopsias = val_biopsias
            else:
                selected_biopsias = test_biopsias

        self.image_paths = []
        for biopsia in selected_biopsias:
            self.image_paths.extend(self.biopsias_dict[biopsia])

        self.image_paths = sorted(self.image_paths)

        paired = []
        missing = []
        for img_path in self.image_paths:
            mask_path = self._get_mask_path(img_path)
            if mask_path.exists():
                paired.append((img_path, mask_path))
            else:
                missing.append((img_path, mask_path))

        if missing:
            warnings.warn(
                f"Found {len(missing)} images without corresponding masks. "
                f"These will be skipped. First few: {missing[:3]}"
            )

        if not paired:
            raise ValueError("No valid image-mask pairs found after checking.")

        self.image_paths, self.mask_paths = zip(*paired)
        self.image_paths = list(self.image_paths)
        self.mask_paths = list(self.mask_paths)
        
        # Cache for positive flags (used by WeightedRandomSampler)
        self._positive_flags = None

    def _get_mask_path(self, image_path: Path) -> Path:
        """Convert */images/<tile>.png to */masks/<tile>_mask.png."""
        rel = image_path.relative_to(self.images_dir)
        parts = list(rel.parts)

        if len(parts) < 3 or parts[1] != 'images':
            raise ValueError(
                f"Unexpected image path layout: {image_path}. "
                "Expected <root>/<biopsia>/images/<tile>.png"
            )

        parts[1] = 'masks'
        stem = Path(parts[-1]).stem
        parts[-1] = f"{stem}_mask.png"
        return self.masks_dir / Path(*parts)

    def get_positive_flags(self) -> list:
        """Return list of bools: True if tile mask contains at least one glomerulus pixel.
        
        Used by WeightedRandomSampler. Reads mask files once at dataset init time.
        Masks are small enough (1024x1024 uint8 = 1MB) that this is feasible.
        Caches result to avoid double scan.
        """
        if self._positive_flags is not None:
            return self._positive_flags
        
        flags = []
        for mask_path in self.mask_paths:
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            if mask is None:
                flags.append(False)
            else:
                flags.append(bool(np.any(mask > 0)))
        self._positive_flags = flags
        return flags

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """Load BGR tile -> Reinhard normalization -> BGR-to-RGB -> Z-score -> augment -> tensors."""
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        img_bgr = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise RuntimeError(f"Failed to load image: {img_path}")

        # Convert BGR to RGB (keep as uint8 for augmentation)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise RuntimeError(f"Failed to load mask: {mask_path}")

        # Binarize: 0 = background, any >0 value = glomerulus
        # Including Excluido (255) as Glomerulus intentionally
        mask_binary = (mask > 0).astype(np.uint8)

        # Apply augmentation BEFORE normalization (on uint8 RGB)
        if self.transforms is not None and self.split == 'train':
            augmented = self.transforms(image=img_rgb, mask=mask_binary)
            img_rgb = augmented['image']
            mask_binary = augmented['mask']

        # Convert back to BGR for Reinhard normalization
        img_bgr_aug = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
        
        # Apply Reinhard normalization
        if self.reinhard_norm is not None:
            img_bgr_aug = self.reinhard_norm(img_bgr_aug)

        # Convert back to RGB for Z-score normalization
        img_rgb = cv2.cvtColor(img_bgr_aug, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

        # Apply Z-score normalization
        if self.channel_means is not None and self.channel_stds is not None:
            for c in range(3):
                img_rgb[..., c] = (img_rgb[..., c] - self.channel_means[c]) / (self.channel_stds[c] + 1e-6)

        img_tensor = torch.from_numpy(np.transpose(img_rgb, (2, 0, 1))).float()
        mask_tensor = torch.from_numpy(mask_binary.astype(np.int64)).long()

        return img_tensor, mask_tensor

In [ ]:
def create_dataloaders(
    images_dir: str,
    masks_dir: str = None,
    batch_size: int = 4,
    num_workers: int = 4,
    seed: int = 42,
    train_transforms=None,
    val_transforms=None,
    reinhard_norm=None,
    channel_means: list = None,
    channel_stds: list = None,
):
    """Create train/val/test DataLoaders with pre-computed grouped splits by biopsia and positive/negative balancing."""
    train_biopsias, val_biopsias, test_biopsias, biopsias_dict = split_biopsias(
        images_dir=images_dir,
        train_size=0.70,
        val_size=0.15,
        seed=seed,
    )

    train_ds = GlomeruliDataset(
        images_dir,
        masks_dir,
        split='train',
        biopsias=train_biopsias,
        biopsias_dict=biopsias_dict,
        reinhard_norm=reinhard_norm,
        channel_means=channel_means,
        channel_stds=channel_stds,
        transforms=train_transforms,
    )

    val_ds = GlomeruliDataset(
        images_dir,
        masks_dir,
        split='val',
        biopsias=val_biopsias,
        biopsias_dict=biopsias_dict,
        reinhard_norm=reinhard_norm,
        channel_means=channel_means,
        channel_stds=channel_stds,
        transforms=val_transforms,
    )

    test_ds = GlomeruliDataset(
        images_dir,
        masks_dir,
        split='test',
        biopsias=test_biopsias,
        biopsias_dict=biopsias_dict,
        reinhard_norm=reinhard_norm,
        channel_means=channel_means,
        channel_stds=channel_stds,
        transforms=None,
    )

    # ========== CHANGE 3: WeightedRandomSampler for positive/negative balance ==========
    print("Computing positive/negative tile flags for WeightedRandomSampler...")
    positive_flags = train_ds.get_positive_flags()
    n_positive = sum(positive_flags)
    n_negative = len(positive_flags) - n_positive
    print(f"  Positive tiles (contain glomerulus): {n_positive:,}")
    print(f"  Negative tiles (background only):    {n_negative:,}")

    if n_positive == 0 or n_negative == 0:
        warnings.warn(
            f"Cannot balance sampler: positives={n_positive}, negatives={n_negative}. "
            "Falling back to normal shuffle."
        )
        sampler = None
        shuffle_train = True
    else:
        weight_pos = 1.0 / n_positive
        weight_neg = 1.0 / n_negative
        sample_weights = torch.tensor(
            [weight_pos if flag else weight_neg for flag in positive_flags],
            dtype=torch.float32,
        )
        num_samples = min(len(positive_flags), 2 * n_positive)
        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=num_samples,
            replacement=True,
        )
        shuffle_train = False
        print(f"  WeightedRandomSampler: {num_samples:,} samples/epoch (~1:1 pos:neg ratio)")

    train_loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=shuffle_train,
        sampler=sampler,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=(num_workers > 0),
        prefetch_factor=3 if num_workers > 0 else None,
    )

    val_loader = torch.utils.data.DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=(num_workers > 0),
        prefetch_factor=2 if num_workers > 0 else None,
    )

    test_loader = torch.utils.data.DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=(num_workers > 0),
        prefetch_factor=2 if num_workers > 0 else None,
    )

    return train_loader, val_loader, test_loader

In [ ]:
# Quick test of dataset loading
if Path('Salidas/Tiles_UNet').exists():
    ds_test = GlomeruliDataset('Salidas/Tiles_UNet', split='train')
    print(f"Dataset loaded: {len(ds_test)} tiles")
    print(f"First image path: {ds_test.image_paths[0]}")
    print(f"First mask path:  {ds_test.mask_paths[0]}")

    img, mask = ds_test[0]
    print(f"Image shape: {img.shape}, dtype: {img.dtype}")
    print(f"Mask shape: {mask.shape}, dtype: {mask.dtype}")
    print(f"Mask unique classes (binary): {torch.unique(mask).tolist()}")
    print(f"Class distribution: 0 (background)={torch.sum(mask == 0).item()}, 1 (glomerulus)={torch.sum(mask == 1).item()}")

    positive_idx = next((i for i, p in enumerate(ds_test.mask_paths) if cv2.imread(str(p), cv2.IMREAD_GRAYSCALE).max() > 0), None)
    if positive_idx is not None:
        _, positive_mask = ds_test[positive_idx]
        print(f"Positive mask sanity check at idx={positive_idx}: classes={torch.unique(positive_mask).tolist()}, "
              f"glomerulus_pixels={torch.sum(positive_mask == 1).item()}")
    else:
        warnings.warn("No positive masks found in this split. Check tiling annotations or split selection.")
else:
    print("Dataset directory not found. Run preprocessing pipeline first.")
    print("Required: tiling_unet.py -> normalizacion.py")



In [ ]:
class BinaryDiceLoss(nn.Module):
    """Sørensen-Dice coefficient for 1-channel binary logits (sigmoid output)."""

    def __init__(self, smooth: float = 1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Args:
            logits: [B, 1, H, W] raw logits from smp model
            targets: [B, H, W] int64 binary labels {0, 1}
        """
        probs = torch.sigmoid(logits).squeeze(1)           # [B, H, W]
        targets_f = targets.float()                        # [B, H, W]
        
        intersection = (probs * targets_f).sum(dim=(1, 2))
        cardinality   = probs.sum(dim=(1, 2)) + targets_f.sum(dim=(1, 2))
        
        dice_per_sample = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        return 1.0 - dice_per_sample.mean()

In [ ]:
class BinaryCombinedLoss(nn.Module):
    """BCE-with-logits + Dice loss for binary segmentation (1-channel output)."""

    def __init__(
        self,
        weight_bce: float = 0.5,
        weight_dice: float = 0.5,
        smooth: float = 1e-6,
    ):
        super().__init__()
        self.weight_bce = weight_bce
        self.weight_dice = weight_dice
        self.dice_loss = BinaryDiceLoss(smooth=smooth)
    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Args:
            logits:  [B, 1, H, W] raw logits from smp model
            targets: [B, H, W] int64 binary labels {0, 1}
        """
        targets_f = targets.float().unsqueeze(1)           # [B, 1, H, W]
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets_f)
        dice = self.dice_loss(logits, targets)
        return self.weight_bce * bce + self.weight_dice * dice


In [ ]:
# Smoke test for binary loss functions (1-channel logits)
B, H, W = 2, 256, 256
logits = torch.randn(B, 1, H, W, device=device, requires_grad=True)
targets = torch.randint(0, 2, (B, H, W), device=device)

dice_loss = BinaryDiceLoss()
dice = dice_loss(logits, targets)
print(f"Binary Dice Loss: {dice.item():.4f}")

combined = BinaryCombinedLoss()
loss = combined(logits, targets)
print(f"Binary Combined Loss: {loss.item():.4f}")

loss.backward()
print("✓ Backward pass OK")

## Part 3: Model — segmentation_models_pytorch U-Net with ResNet-34 backbone

The executable model definition is in the next code cell. It uses `segmentation_models_pytorch.Unet` with a ResNet-34 encoder, ImageNet encoder weights, 3-channel RGB input, and a single binary logit output channel (`sigmoid > 0.5` at inference time).


In [ ]:
# Model configuration and factory
MODEL_CONFIG = {
    'encoder_name': 'resnet34',
    'encoder_weights': 'imagenet',
    'in_channels': 3,
    'classes': 1,  # Single binary channel (sigmoid)
}

def create_model(
    encoder_name: str = MODEL_CONFIG['encoder_name'],
    encoder_weights: str = MODEL_CONFIG['encoder_weights'],
    in_channels: int = MODEL_CONFIG['in_channels'],
    classes: int = MODEL_CONFIG['classes'],
) -> smp.Unet:
    """Create smp.Unet with binary output (1 channel logits, no activation)."""
    model = smp.Unet(
        encoder_name=encoder_name,
        encoder_weights=encoder_weights,
        in_channels=in_channels,
        classes=classes,
        activation=None,  # Raw logits for BCE-with-logits loss
    )
    return model


In [ ]:
# Smoke test: smp U-Net forward pass
model_test = create_model().to(device)
total_params = sum(p.numel() for p in model_test.parameters())
encoder_params = sum(p.numel() for p in model_test.encoder.parameters())
decoder_params = total_params - encoder_params

print(f"smp U-Net total parameters:   {total_params:,}")
print(f"  Encoder (ResNet-34):         {encoder_params:,}")
print(f"  Decoder + head:              {decoder_params:,}")

x_test = torch.randn(1, 3, 1024, 1024, device=device)
with torch.no_grad():
    out_test = model_test(x_test)
print(f"Input shape:  {x_test.shape}")
print(f"Output shape: {out_test.shape}")
assert out_test.shape == (1, 1, 1024, 1024), f"Expected (1, 1, 1024, 1024), got {out_test.shape}"
print("✓ smp U-Net forward pass OK")

del model_test, x_test, out_test
torch.cuda.empty_cache()

## Part 4: Training Pipeline

Complete training system with:
- Dynamic DataLoader worker calculation (based on available RAM)
- Binary Segmentation Metrics (Accuracy, Precision, Recall, F1, ROC-AUC, Dice)
- Augmentation pipelines (train-specific)
- Optimizer: AdamW with weight decay
- LR Scheduler: Linear warmup (5 epochs) -> Cosine annealing
- Checkpointing: saves best model (by Val F1) + last model
- TensorBoard logging

### Training Workflow

1. Load data (train/val/test splits grouped by biopsia)
2. Create model, optimizer, scheduler
3. Train for N epochs:
   - Forward pass on batch
   - Compute loss (BCEWithLogits + Dice for binary segmentation)
   - Backward pass + gradient clipping
   - Optimizer step
   - Validate after each epoch (compute binary metrics on val set)
   - Save checkpoint if best model found
4. Final evaluation on test set
5. Save metrics report

In [ ]:
class MeanIoUMetric:
    """IoU for binary segmentation with 1-channel sigmoid output."""

    def __init__(self, threshold: float = 0.5):
        self.threshold = threshold
        self.reset()

    def reset(self):
        self.intersection = 0
        self.union = 0

    def update(self, logits: torch.Tensor, target: torch.Tensor):
        """
        Args:
            logits:  [B, 1, H, W] raw logits
            target:  [B, H, W] int64 binary labels
        """
        probs = torch.sigmoid(logits).squeeze(1)           # [B, H, W]
        pred_labels = (probs > self.threshold).long()

        pred_np = pred_labels.cpu().numpy()
        tgt_np  = target.cpu().numpy()

        self.intersection += np.logical_and(pred_np == 1, tgt_np == 1).sum()
        self.union        += np.logical_or(pred_np == 1, tgt_np == 1).sum()

    def compute(self) -> float:
        if self.union == 0:
            return 0.0
        return float(self.intersection / self.union)

In [ ]:
class BinarySegmentationMetrics:
    """Accuracy, Precision, Recall, F1, Dice, and ROC-AUC for binary segmentation with sigmoid output."""

    def __init__(self, max_auc_pixels: int = 100000, threshold: float = 0.5):
        """Store subsampled pixels to prevent memory explosion with large images."""
        self.max_auc_pixels = max_auc_pixels
        self.threshold = threshold
        self.reset()

    def reset(self):
        self.tp = 0
        self.tn = 0
        self.fp = 0
        self.fn = 0
        
        self.all_probs = []
        self.all_targets = []
        self.total_auc_pixels = 0

    def update(self, pred: torch.Tensor, target: torch.Tensor):
        """Accumulate confusion matrix and subsampled pixel predictions.
        
        Args:
            pred: [B, 1, H, W] raw logits from smp model
            target: [B, H, W] int64 binary labels {0, 1}
        """
        prob_pos = torch.sigmoid(pred).squeeze(1)          # [B, H, W]
        pred_labels = (prob_pos > self.threshold).long()

        pred_labels = pred_labels.cpu().numpy()
        target = target.cpu().numpy()
        prob_pos = prob_pos.cpu().numpy()

        pred_flat = pred_labels.flatten()
        target_flat = target.flatten()
        prob_flat = prob_pos.flatten()

        self.tp += np.logical_and(target_flat == 1, pred_flat == 1).sum()
        self.tn += np.logical_and(target_flat == 0, pred_flat == 0).sum()
        self.fp += np.logical_and(target_flat == 0, pred_flat == 1).sum()
        self.fn += np.logical_and(target_flat == 1, pred_flat == 0).sum()

        budget = self.max_auc_pixels - len(self.all_probs)
        if budget > 0:
            if len(prob_flat) <= budget:
                self.all_probs.extend(prob_flat)
                self.all_targets.extend(target_flat)
            else:
                idx = np.random.choice(len(prob_flat), size=budget, replace=False)
                self.all_probs.extend(prob_flat[idx])
                self.all_targets.extend(target_flat[idx])
        
        self.total_auc_pixels += len(prob_flat)

    def compute(self) -> dict:
        """Return accuracy, precision, recall, f1, dice, roc_auc."""
        metrics = {}

        total = self.tp + self.tn + self.fp + self.fn
        if total > 0:
            metrics['accuracy'] = float((self.tp + self.tn) / total)
        else:
            metrics['accuracy'] = 0.0

        if (self.tp + self.fp) > 0:
            metrics['precision'] = float(self.tp / (self.tp + self.fp))
        else:
            metrics['precision'] = 0.0

        if (self.tp + self.fn) > 0:
            metrics['recall'] = float(self.tp / (self.tp + self.fn))
        else:
            metrics['recall'] = 0.0

        if (metrics['precision'] + metrics['recall']) > 0:
            metrics['f1'] = float(
                2 * (metrics['precision'] * metrics['recall']) / 
                (metrics['precision'] + metrics['recall'])
            )
        else:
            metrics['f1'] = 0.0

        if (2 * self.tp + self.fp + self.fn) > 0:
            metrics['dice_metric'] = float(
                (2 * self.tp) / (2 * self.tp + self.fp + self.fn + 1e-8)
            )
        else:
            metrics['dice_metric'] = 0.0

        if len(self.all_targets) > 0 and len(set(self.all_targets)) > 1:
            try:
                from sklearn.metrics import roc_auc_score
                auc = roc_auc_score(self.all_targets, self.all_probs)
                metrics['roc_auc'] = float(auc)
            except Exception as e:
                print(f"Warning: Could not compute ROC-AUC: {e}")
                metrics['roc_auc'] = 0.0
        else:
            metrics['roc_auc'] = 0.0

        return metrics

In [ ]:
def get_transforms(reinhard_norm=None, channel_means=None, channel_stds=None, size: int = 1024):
    """Return augmentation pipelines for train/val.
    
    Applied on uint8 RGB numpy arrays BEFORE Reinhard and Z-score normalization.
    Normalization happens later in GlomeruliDataset.__getitem__.
    """
    train_augment = A.Compose([
        # Geometric augmentations — applied to both image AND mask
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.75),
        A.Transpose(p=0.5),
        A.Rotate(limit=15, p=0.3),
        
        # Elastic deformations — simulate tissue preparation artifacts
        A.ElasticTransform(alpha=120, sigma=120 * 0.05, p=0.3),
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.2),
        
        # Color augmentations — simulate stain variation between slides
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.3),
        
        # Noise — simulate scanner artifacts
        A.GaussNoise(std_range=(0.01, 0.05), p=0.2),
        
        # Regularization via occlusion (use fill=128 to avoid NaN in BatchNorm)
        A.CoarseDropout(
            num_holes_range=(1, 8),
            hole_height_range=(32, 64),
            hole_width_range=(32, 64),
            fill=128,
            p=0.2,
        ),
    ], additional_targets={'mask': 'mask'})
    
    val_transform = A.Compose([])
    return train_augment, val_transform

In [ ]:
def train_epoch(
    model: nn.Module,
    dataloader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: torch.device,
    scaler: GradScaler = None,
    epoch: int = 0,
    total_epochs: int = 1,
) -> float:
    """Train one epoch with tqdm progress, per-batch timing, and AMP (FP16)."""
    model.train()
    total_loss = 0.0
    num_batches = 0
    running_dice = 0.0
    
    epoch_start = time.time()
    
    pbar = tqdm(
        dataloader,
        desc=f"Epoch {epoch+1}/{total_epochs} [train]",
        unit="batch",
        dynamic_ncols=True,
        leave=True,
    )
    
    for batch_idx, (images, masks) in enumerate(pbar):
        t_data_end = time.time()

        images = images.to(device, non_blocking=True)
        masks  = masks.to(device, non_blocking=True)

        t_transfer_end = time.time()

        optimizer.zero_grad(set_to_none=True)

        t_forward_start = time.time()
        if scaler is not None:
            with autocast():
                logits = model(images)
                loss = criterion(logits, masks)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images)
            loss = criterion(logits, masks)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        t_batch_end = time.time()

        batch_loss = loss.item()
        total_loss += batch_loss
        num_batches += 1

        # Quick inline Dice for the progress bar (no grad, cheap)
        with torch.no_grad():
            prob = torch.sigmoid(logits).squeeze(1)
            pred = (prob > 0.5).float()
            tgt  = masks.float()
            inter = (pred * tgt).sum()
            denom = pred.sum() + tgt.sum()
            batch_dice = float((2 * inter + 1e-6) / (denom + 1e-6))
        running_dice = (running_dice * (num_batches - 1) + batch_dice) / num_batches

        pbar.set_postfix({
            'loss': f'{batch_loss:.4f}',
            'avg_loss': f'{total_loss/num_batches:.4f}',
            'dice': f'{batch_dice:.3f}',
            's/batch': f'{t_batch_end - t_forward_start:.2f}',
        })

        # Print first batch immediately so we know training started
        if batch_idx == 0:
            print(f"\n  [Epoch {epoch+1}] First batch: loss={batch_loss:.4f} | "
                  f"dice={batch_dice:.3f} | "
                  f"data_transfer={t_transfer_end - t_data_end:.3f}s | "
                  f"forward+backward={t_batch_end - t_forward_start:.3f}s")

    epoch_elapsed = time.time() - epoch_start
    avg_loss = total_loss / num_batches if num_batches > 0 else 0.0
    print(f"  Epoch {epoch+1} train done: avg_loss={avg_loss:.4f} | "
          f"avg_dice={running_dice:.3f} | elapsed={epoch_elapsed:.1f}s")
    return avg_loss

In [ ]:
def eval_epoch(
    model: nn.Module,
    dataloader,
    criterion: nn.Module,
    device: torch.device,
    split_name: str = 'val',
) -> Tuple[float, dict]:
    """Evaluate one epoch with tqdm progress. Returns loss and binary segmentation metrics."""
    model.eval()
    total_loss = 0.0
    num_batches = 0
    seg_metrics = BinarySegmentationMetrics()
    iou_metric  = MeanIoUMetric(threshold=0.5)

    pbar = tqdm(dataloader, desc=f"  [{split_name}]", unit="batch",
                dynamic_ncols=True, leave=False)

    with torch.no_grad():
        for images, masks in pbar:
            images = images.to(device, non_blocking=True)
            masks  = masks.to(device, non_blocking=True)

            with autocast():
                logits = model(images)
                loss   = criterion(logits, masks)

            total_loss += loss.item()
            num_batches += 1

            seg_metrics.update(logits, masks)
            iou_metric.update(logits, masks)

            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = total_loss / num_batches if num_batches > 0 else 0.0
    metrics  = seg_metrics.compute()
    metrics['mean_iou'] = iou_metric.compute()

    return avg_loss, metrics

In [ ]:
def compute_dataloader_workers(batch_size: int = 8, max_workers: int = 8) -> int:
    """Heuristic: calculate optimal DataLoader workers based on available CPU cores."""
    import os

    n_cpus = os.cpu_count() or 4
    n_workers = min(max(batch_size // 2, 2), n_cpus, max_workers)
    return n_workers


In [ ]:
def audit_mask_distribution(
    datasets: dict,
    expected_values=None,
    almost_full_threshold: float = 0.80,
):
    """Print a quantitative mask audit before training."""
    print("\n--- Mask Audit Report (pre-training) ---")
    if expected_values is None:
        print("Expected raw mask values: not enforced")
    else:
        expected_values = frozenset(expected_values)
        print(f"Expected raw mask values: {sorted(expected_values)}")
    print(f"Almost-full tile threshold: positive_area_ratio >= {almost_full_threshold:.2f}")

    all_unexpected_values = set()
    audit_summary = {}
    percentiles = [0, 1, 5, 25, 50, 75, 95, 99, 100]

    for split_name, dataset in datasets.items():
        tile_records = []
        value_pixel_counts = {}
        value_tile_counts = {}
        slide_stats = {}

        for idx, mask_path in enumerate(dataset.mask_paths):
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            if mask is None:
                warnings.warn(f"Could not read mask during audit: {mask_path}")
                continue

            image_path = dataset.image_paths[idx]
            slide = _slide_name_from_image_path(image_path, dataset.images_dir)
            get_tile_metadata = getattr(dataset, 'get_tile_metadata', None)
            if callable(get_tile_metadata):
                try:
                    meta = get_tile_metadata(idx)
                    slide = meta.get('slide') or slide
                except Exception:
                    pass

            unique_vals, counts = np.unique(mask, return_counts=True)
            unique_vals = [int(v) for v in unique_vals]
            counts = [int(c) for c in counts]
            for val, count in zip(unique_vals, counts):
                value_pixel_counts[val] = value_pixel_counts.get(val, 0) + count
                value_tile_counts[val] = value_tile_counts.get(val, 0) + 1

            if expected_values is not None:
                unexpected_vals = set(unique_vals) - set(expected_values)
                all_unexpected_values.update(unexpected_vals)
            else:
                unexpected_vals = set()

            pos_mask = mask > 0
            pos_pixels = int(pos_mask.sum())
            total_pixels = int(mask.size)
            neg_pixels = total_pixels - pos_pixels
            pos_ratio = pos_pixels / total_pixels if total_pixels else 0.0
            is_positive = pos_pixels > 0
            is_almost_full = pos_ratio >= almost_full_threshold

            tile_records.append({
                'slide': slide,
                'pos_pixels': pos_pixels,
                'neg_pixels': neg_pixels,
                'pos_ratio': pos_ratio,
                'is_positive': is_positive,
                'is_empty': not is_positive,
                'is_almost_full': is_almost_full,
                'unique_values': unique_vals,
                'unexpected_values': sorted(unexpected_vals),
                'mask_path': str(mask_path),
            })

            st = slide_stats.setdefault(slide, {
                'tiles': 0,
                'positive_tiles': 0,
                'empty_tiles': 0,
                'almost_full_tiles': 0,
                'pos_pixels': 0,
                'neg_pixels': 0,
                'values': {},
            })
            st['tiles'] += 1
            st['positive_tiles'] += int(is_positive)
            st['empty_tiles'] += int(not is_positive)
            st['almost_full_tiles'] += int(is_almost_full)
            st['pos_pixels'] += pos_pixels
            st['neg_pixels'] += neg_pixels
            for val, count in zip(unique_vals, counts):
                st['values'][val] = st['values'].get(val, 0) + count

        if not tile_records:
            warnings.warn(f"No readable masks found for split '{split_name}' during audit.")
            audit_summary[split_name] = {'tile_records': [], 'slide_stats': {}, 'value_pixel_counts': {}}
            continue

        ratios = np.array([r['pos_ratio'] for r in tile_records], dtype=np.float64)
        positive_ratios = np.array([r['pos_ratio'] for r in tile_records if r['is_positive']], dtype=np.float64)
        all_pct = {p: float(np.percentile(ratios, p)) for p in percentiles}
        pos_pct = {p: float(np.percentile(positive_ratios, p)) for p in percentiles} if positive_ratios.size else {}

        n_tiles = len(tile_records)
        n_positive = sum(r['is_positive'] for r in tile_records)
        n_empty = sum(r['is_empty'] for r in tile_records)
        n_almost_full = sum(r['is_almost_full'] for r in tile_records)
        pos_pixels_total = sum(r['pos_pixels'] for r in tile_records)
        neg_pixels_total = sum(r['neg_pixels'] for r in tile_records)
        total_pixels = pos_pixels_total + neg_pixels_total
        raw_values_seen = sorted(value_pixel_counts)
        unexpected_seen = sorted(set(raw_values_seen) - set(expected_values)) if expected_values is not None else []
        rare_values = {
            val: {'tiles': value_tile_counts[val], 'pixels': value_pixel_counts[val]}
            for val in raw_values_seen
            if val != 0 and value_tile_counts[val] <= max(1, int(0.01 * n_tiles))
        }

        print(f"\n[{split_name.upper()}]")
        print(f"  Tiles: {n_tiles:,} | positive: {n_positive:,} | empty: {n_empty:,} | almost full: {n_almost_full:,}")
        print(f"  Pixels: positive={pos_pixels_total:,} negative={neg_pixels_total:,} "
              f"positive_ratio={(pos_pixels_total / total_pixels if total_pixels else 0):.6f}")
        print("  Positive area ratio percentiles (all tiles): " +
              ", ".join(f"p{p}={all_pct[p]:.6f}" for p in percentiles))
        if pos_pct:
            print("  Positive area ratio percentiles (positive tiles only): " +
                  ", ".join(f"p{p}={pos_pct[p]:.6f}" for p in percentiles))
        else:
            print("  Positive area ratio percentiles (positive tiles only): no positive tiles")
        print(f"  Raw mask values seen: {raw_values_seen}")
        print(f"  Raw mask pixel counts: {dict(sorted(value_pixel_counts.items()))}")
        print(f"  Raw mask tile counts: {dict(sorted(value_tile_counts.items()))}")
        if unexpected_seen:
            warnings.warn(f"Unexpected raw mask values in {split_name}: {unexpected_seen}")
        if rare_values:
            print(f"  Rare non-zero values (<=1% of tiles): {rare_values}")

        print("  Per-slide summary:")
        for slide, st in sorted(slide_stats.items()):
            slide_total = st['pos_pixels'] + st['neg_pixels']
            slide_pos_ratio = st['pos_pixels'] / slide_total if slide_total else 0.0
            flags = []
            if st['positive_tiles'] == 0:
                flags.append('NO_POSITIVES')
            if st['almost_full_tiles'] > 0:
                flags.append('ALMOST_FULL_TILES')
            flag_txt = f" [{' | '.join(flags)}]" if flags else ""
            print(
                f"    {slide}: tiles={st['tiles']:,}, positive_tiles={st['positive_tiles']:,}, "
                f"empty_tiles={st['empty_tiles']:,}, almost_full={st['almost_full_tiles']:,}, "
                f"pos_pixels={st['pos_pixels']:,}, neg_pixels={st['neg_pixels']:,}, "
                f"pos_ratio={slide_pos_ratio:.6f}, values={sorted(st['values'])}{flag_txt}"
            )

        audit_summary[split_name] = {
            'tile_records': tile_records,
            'slide_stats': slide_stats,
            'value_pixel_counts': value_pixel_counts,
            'value_tile_counts': value_tile_counts,
            'unexpected_values': unexpected_seen,
            'rare_values': rare_values,
            'percentiles_all_tiles': all_pct,
            'percentiles_positive_tiles': pos_pct,
        }

    if all_unexpected_values:
        warnings.warn(f"Unexpected raw mask values found across splits: {sorted(all_unexpected_values)}")
    print("\n--- End Mask Audit Report ---")
    return audit_summary


def train_unet(
    epochs: int = 50,
    batch_size: int = 8,
    lr: float = 1e-3,
    images_dir: str = 'Salidas/Tiles_UNet',
    output_dir: str = 'checkpoints',
    num_workers: int = None,
    seed: int = 42,
    warmup_epochs: int = 5,
    use_amp: bool = True,
):
    """Complete training pipeline with smp.Unet, WeightedRandomSampler, and tqdm logging."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    run_name = f"unet_binary_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    log_dir = output_dir / run_name
    writer = SummaryWriter(str(log_dir))

    print(f"Run: {run_name}")
    print(f"Device: {device}")
    print(f"AMP (FP16): {use_amp and device.type == 'cuda'}")
    print(f"Output dir: {output_dir}")

    print("\n--- Data Loading Pipeline ---")
    print("Step 1: Computing grouped split by biopsia...")
    train_biopsias, val_biopsias, test_biopsias, biopsias_dict = split_biopsias(
        images_dir=images_dir,
        train_size=0.70,
        val_size=0.15,
        seed=seed,
    )
    print(f"  Train biopsies: {len(train_biopsias)}")
    print(f"  Val biopsies: {len(val_biopsias)}")
    print(f"  Test biopsies: {len(test_biopsias)}")

    print("\nStep 2: Collecting train image paths...")
    train_image_paths = []
    for biopsia in train_biopsias:
        train_image_paths.extend(biopsias_dict[biopsia])
    print(f"  Train tiles: {len(train_image_paths)}")

    print("\nStep 3: Computing Reinhard stain normalization template...")
    try:
        reinhard_stats = ReinhardNormalize.compute_template_stats(train_image_paths, n_samples=200)
        reinhard_norm = ReinhardNormalize(reinhard_stats)
        print(f"  Template stats computed from ~{min(200, len(train_image_paths))} samples")
        print(f"    L: mean={reinhard_stats['mean_L']:.1f}, std={reinhard_stats['std_L']:.1f}")
        print(f"    a: mean={reinhard_stats['mean_a']:.1f}, std={reinhard_stats['std_a']:.1f}")
        print(f"    b: mean={reinhard_stats['mean_b']:.1f}, std={reinhard_stats['std_b']:.1f}")
    except Exception as e:
        print(f"  Warning: Could not compute Reinhard stats: {e}")
        reinhard_norm = None
        reinhard_stats = None

    print("\nStep 4: Computing per-channel Z-score normalization stats...")
    try:
        channel_means, channel_stds = compute_channel_stats(train_image_paths, n_samples=200)
        print(f"  Channel means (RGB): {[f'{m:.4f}' for m in channel_means]}")
        print(f"  Channel stds (RGB):  {[f'{s:.4f}' for s in channel_stds]}")
    except Exception as e:
        print(f"  Warning: Could not compute channel stats: {e}")
        channel_means = None
        channel_stds = None

    print("\nStep 5: Creating augmentation pipelines...")
    train_transforms, val_transforms = get_transforms(
        reinhard_norm=reinhard_norm,
        channel_means=channel_means,
        channel_stds=channel_stds,
    )

    if num_workers is None:
        num_workers = compute_dataloader_workers(batch_size=batch_size)
        print(f"  Auto-calculated DataLoader workers: {num_workers}")
    else:
        print(f"  Using specified DataLoader workers: {num_workers}")

    print("\nStep 5b: Creating dataloaders with pre-computed splits and WeightedRandomSampler...")
    train_loader, val_loader, test_loader = create_dataloaders(
        images_dir=images_dir,
        batch_size=batch_size,
        num_workers=num_workers,
        seed=seed,
        train_transforms=train_transforms,
        val_transforms=val_transforms,
        reinhard_norm=reinhard_norm,
        channel_means=channel_means,
        channel_stds=channel_stds,
    )

    print(f"  Train: {len(train_loader.dataset)} tiles | "
          f"Val: {len(val_loader.dataset)} tiles | "
          f"Test: {len(test_loader.dataset)} tiles")

    audit_summary = audit_mask_distribution({
        'train': train_loader.dataset,
        'val': val_loader.dataset,
        'test': test_loader.dataset,
    })

    print("\nCreating smp.Unet model with ResNet-34 encoder...")
    model = create_model().to(device)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {total_params:,}")

    criterion = BinaryCombinedLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    scaler = GradScaler() if (use_amp and device.type == 'cuda') else None
    if scaler:
        print("AMP (FP16) enabled — GradScaler initialized")

    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[
            LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs),
            CosineAnnealingLR(optimizer, T_max=epochs - warmup_epochs, eta_min=1e-6),
        ],
        milestones=[warmup_epochs],
    )

    threshold = 0.5
    best_val_f1 = -1.0
    train_history = []

    print("\nStarting training...")
    for epoch in range(epochs):
        print(f"\n{'='*60}")
        print(f"Epoch [{epoch+1}/{epochs}]")
        print(f"{'='*60}")

        train_loss = train_epoch(
            model, train_loader, criterion, optimizer, device,
            scaler=scaler, epoch=epoch, total_epochs=epochs,
        )

        val_loss, val_metrics = eval_epoch(model, val_loader, criterion, device, split_name='val')
        print(f"Val Loss: {val_loss:.4f}")
        print(f"Val Metrics:")
        print(f"  Accuracy:  {val_metrics['accuracy']:.4f}")
        print(f"  Precision: {val_metrics['precision']:.4f}")
        print(f"  Recall:    {val_metrics['recall']:.4f}")
        print(f"  F1:        {val_metrics['f1']:.4f}")
        print(f"  Dice:      {val_metrics.get('dice_metric', 0.0):.4f}")
        print(f"  mIoU:      {val_metrics.get('mean_iou', 0.0):.4f}")
        print(f"  ROC-AUC:   {val_metrics['roc_auc']:.4f}")

        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        print(f"LR: {current_lr:.2e}")

        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict() if scaler else None,
            'best_val_f1': best_val_f1,
            'channel_means': channel_means,
            'channel_stds': channel_stds,
            'reinhard_stats': reinhard_stats if reinhard_norm else None,
            'model_config': MODEL_CONFIG.copy(),
            'threshold': threshold,
        }

        torch.save(checkpoint, output_dir / f'{run_name}_last.pth')

        if epoch == 0 or val_metrics['f1'] > best_val_f1:
            best_val_f1 = val_metrics['f1']
            checkpoint['best_val_f1'] = best_val_f1
            torch.save(checkpoint, output_dir / f'{run_name}_best.pth')
            print(f"✓ Best model saved! (Val F1: {val_metrics['f1']:.4f})")

        writer.add_scalar('loss/train', train_loss, epoch)
        writer.add_scalar('loss/val', val_loss, epoch)
        writer.add_scalar('metric/val_accuracy', val_metrics['accuracy'], epoch)
        writer.add_scalar('metric/val_precision', val_metrics['precision'], epoch)
        writer.add_scalar('metric/val_recall', val_metrics['recall'], epoch)
        writer.add_scalar('metric/val_f1', val_metrics['f1'], epoch)
        writer.add_scalar('metric/val_dice_metric', val_metrics.get('dice_metric', 0.0), epoch)
        writer.add_scalar('metric/val_mean_iou', val_metrics.get('mean_iou', 0.0), epoch)
        writer.add_scalar('metric/val_roc_auc', val_metrics['roc_auc'], epoch)
        writer.add_scalar('lr', current_lr, epoch)

        train_history.append({
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_metrics': val_metrics,
        })

    writer.close()

    print(f"\n{'='*60}")
    print("Final evaluation on test set...")
    print(f"{'='*60}")

    best_ckpt = torch.load(output_dir / f'{run_name}_best.pth', map_location=device)
    model.load_state_dict(best_ckpt['model_state_dict'])

    test_loss, test_metrics = eval_epoch(model, test_loader, criterion, device, split_name='test')
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Metrics:")
    print(f"  Accuracy:  {test_metrics['accuracy']:.4f}")
    print(f"  Precision: {test_metrics['precision']:.4f}")
    print(f"  Recall:    {test_metrics['recall']:.4f}")
    print(f"  F1:        {test_metrics['f1']:.4f}")
    print(f"  Dice:      {test_metrics.get('dice_metric', 0.0):.4f}")
    print(f"  mIoU:      {test_metrics.get('mean_iou', 0.0):.4f}")
    print(f"  ROC-AUC:   {test_metrics['roc_auc']:.4f}")

    report = {
        'run_name': run_name,
        'model_config': MODEL_CONFIG.copy(),
        'training_config': {
            'epochs': epochs,
            'batch_size': batch_size,
            'learning_rate': lr,
            'warmup_epochs': warmup_epochs,
            'use_amp': use_amp,
        },
        'normalization_params': {
            'reinhard_stats': reinhard_stats if reinhard_norm else None,
            'channel_means': channel_means,
            'channel_stds': channel_stds,
            'threshold': threshold,
        },
        'final_metrics': {
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_metrics': val_metrics,
            'test_loss': test_loss,
            'test_metrics': test_metrics,
        },
        'training_history': train_history,
    }

    with open(output_dir / f'{run_name}_report.json', 'w') as f:
        json.dump(report, f, indent=2)

    print(f"\n✓ Training complete!")
    print(f"Checkpoints saved to: {output_dir}")
    print(f"TensorBoard logs: tensorboard --logdir {log_dir}")
    
    return model, report

## Part 6: Usage Instructions

### Running Training

Uncomment and run the cell below to start training:

In [ ]:
if Path('Salidas/Tiles_UNet').exists():
    model, report = train_unet(
        epochs=50,
        batch_size=8,          # Optimized for T4 15.93 GB VRAM with AMP
        lr=1e-3,
        images_dir='Salidas/Tiles_UNet',
        output_dir='checkpoints',
        num_workers=None,      # Auto-calculate based on 28 GB RAM
        seed=42,
        warmup_epochs=5,
        use_amp=True,          # AMP (FP16) for T4 Tensor Cores — 2–3× speedup
    )
else:
    print("Run the preprocessing pipeline first:")
    print("  python tiling_unet.py")

In [ ]:
# ============================================================================
# Part 9: Post-processing — Instance NMS for Overlapping Tile Predictions
# ============================================================================
# When inference is done on overlapping tiles (stride < tile_size), the same
# glomerulus appears in multiple tiles. NMS suppresses duplicate detections.

from skimage.measure import label, regionprops

def apply_nms_to_predictions(
    pred_masks: list,          # list of [H, W] binary numpy arrays (one per tile)
    pred_probs: list,          # list of [H, W] float32 numpy arrays (sigmoid probs)
    tile_coords: list,         # list of (x_start, y_start, x_end, y_end) in WSI pixels
    iou_threshold: float = 0.5,
) -> list:
    """Apply NMS across tiles to remove duplicate glomerulus detections from overlapping tiles.
    
    Args:
        pred_masks:   Binary prediction masks per tile (0/1 numpy arrays)
        pred_probs:   Sigmoid probability maps per tile (float32)
        tile_coords:  WSI-level bounding boxes for each tile: (x0, y0, x1, y1)
        iou_threshold: Suppress instances with IoU > this threshold
    
    Returns:
        List of kept (bbox, confidence) tuples representing non-overlapping 
        glomerulus detections in WSI coordinates: ((x0, y0, x1, y1), confidence)
    """
    all_instances = []

    for tile_idx, (bin_mask, prob_map, coords) in enumerate(
        zip(pred_masks, pred_probs, tile_coords)
    ):
        x0_tile, y0_tile, x1_tile, y1_tile = coords
        tile_h = bin_mask.shape[0]
        tile_w = bin_mask.shape[1]

        # Scale factors: tile pixels → WSI pixels
        scale_x = (x1_tile - x0_tile) / tile_w
        scale_y = (y1_tile - y0_tile) / tile_h

        # Connected components
        labeled = label(bin_mask, connectivity=2)
        props = regionprops(labeled, intensity_image=prob_map)

        for region in props:
            if region.area < 100:   # skip tiny noise components (<100 pixels)
                continue

            # Bounding box in tile pixel coords
            r0, c0, r1, c1 = region.bbox  # row0, col0, row1, col1

            # Convert to WSI coordinates
            wsi_x0 = x0_tile + c0 * scale_x
            wsi_y0 = y0_tile + r0 * scale_y
            wsi_x1 = x0_tile + c1 * scale_x
            wsi_y1 = y0_tile + r1 * scale_y

            # Confidence = mean sigmoid probability over component pixels
            confidence = float(region.mean_intensity)

            all_instances.append({
                'bbox': (wsi_x0, wsi_y0, wsi_x1, wsi_y1),
                'confidence': confidence,
                'area': region.area,
            })

    # Sort by confidence descending
    all_instances.sort(key=lambda x: x['confidence'], reverse=True)

    # Greedy NMS
    kept = []
    suppressed = set()

    for i, inst in enumerate(all_instances):
        if i in suppressed:
            continue
        kept.append(inst)

        # Suppress overlapping instances
        bx0, by0, bx1, by1 = inst['bbox']
        for j in range(i + 1, len(all_instances)):
            if j in suppressed:
                continue
            ox0, oy0, ox1, oy1 = all_instances[j]['bbox']

            # Compute IoU
            inter_x0 = max(bx0, ox0)
            inter_y0 = max(by0, oy0)
            inter_x1 = min(bx1, ox1)
            inter_y1 = min(by1, oy1)

            if inter_x1 <= inter_x0 or inter_y1 <= inter_y0:
                iou = 0.0
            else:
                inter_area = (inter_x1 - inter_x0) * (inter_y1 - inter_y0)
                area_i = (bx1 - bx0) * (by1 - by0)
                area_j = (ox1 - ox0) * (oy1 - oy0)
                union_area = area_i + area_j - inter_area
                iou = inter_area / (union_area + 1e-8)

            if iou > iou_threshold:
                suppressed.add(j)

    return [(d['bbox'], d['confidence']) for d in kept]


# Example usage (run after loading best checkpoint):
# pred_masks, pred_probs = [...] # infer on test tiles
# tile_coords = [...] # WSI coordinates for each test tile
# kept_instances = apply_nms_to_predictions(pred_masks, pred_probs, tile_coords, iou_threshold=0.5)
# print(f"Detected {len(kept_instances)} glomeruli after NMS")

## Part 7: Troubleshooting and System Requirements

### Troubleshooting

**"No images found in Salidas/Estandarizados"**
- Run the preprocessing pipeline first (tiling -> normalization -> standardization).

**"CUDA out of memory"**
- Reduce `batch_size` (try 2 or 1)
- Increase `--ram-fraction` in preprocessing scripts
- Use CPU training (slower but uses less VRAM)

**"Expected masks directory"**
- Ensure masks were copied by normalizacion.py and estandarizacion.py to `Salidas/Estandarizados/*/masks/`
- Verify mask naming convention: `{tile_name}_mask.png`

**"No valid image-mask pairs found"**
- Check that image and mask counts match
- Verify paths: images in `*/images/` and masks in `*/masks/`

### System Requirements

- **Python**: 3.10+
- **PyTorch**: 2.0+ (with CUDA if GPU available)
- **RAM**: 16GB minimum, 32GB+ recommended
- **GPU**: Optional but recommended (4GB VRAM minimum)
- **Dependencies**: See requirements.txt (numpy, torch, torchvision, opencv, scikit-learn, albumentations, tensorboard)

### References

- **U-Net**: Ronneberger et al., "U-Net: Convolutional Networks for Biomedical Image Segmentation" (MICCAI 2015)
- **Dice Loss**: Sørensen–Dice coefficient for segmentation evaluation

In [ ]:
def visualize_predictions(
    model,
    image_path,
    mask_path=None,
    device='cpu',
    threshold=0.5,
    reinhard_norm=None,
    channel_means=None,
    channel_stds=None,
):
    """Visualize model predictions on a test tile.
    
    Args:
        model: smp.Unet model (1-channel binary output)
        image_path: Path to input RGB image
        mask_path: Path to ground truth mask (optional)
        device: Torch device
        threshold: Sigmoid threshold for binary classification (default 0.5)
        reinhard_norm: Optional Reinhard normalizer used during training
        channel_means: Optional RGB channel means used during training
        channel_stds: Optional RGB channel stds used during training
    """
    from PIL import Image
    import matplotlib.pyplot as plt
    
    # Load image for display and model preprocessing
    img = Image.open(image_path).convert('RGB')
    img_array = np.array(img, dtype=np.uint8)

    # Match GlomeruliDataset preprocessing: RGB -> BGR -> Reinhard -> RGB/255 -> Z-score
    img_bgr = cv2.cvtColor(img_array, cv2.COLOR_RGB2BGR)
    if reinhard_norm is not None:
        img_bgr = reinhard_norm(img_bgr)

    img_preprocessed = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

    if channel_means is not None and channel_stds is not None:
        for c in range(3):
            img_preprocessed[..., c] = (
                img_preprocessed[..., c] - channel_means[c]
            ) / (channel_stds[c] + 1e-6)
    
    img_tensor = torch.from_numpy(np.transpose(img_preprocessed, (2, 0, 1))).float().unsqueeze(0)
    img_tensor = img_tensor.to(device)
    
    # Model forward pass
    model.eval()
    with torch.no_grad():
        logits = model(img_tensor)  # [1, 1, H, W]
        probs = torch.sigmoid(logits).squeeze(0).squeeze(0).cpu().numpy()  # [H, W]
    
    pred_mask = (probs > threshold).astype(np.uint8) * 255
    
    # Visualization
    n_cols = 3 if mask_path else 2
    fig, axes = plt.subplots(1, n_cols, figsize=(15, 5))
    
    axes[0].imshow(img_array)
    axes[0].set_title('Input Image')
    axes[0].axis('off')
    
    axes[1].imshow(pred_mask, cmap='gray')
    axes[1].set_title(f'Prediction (threshold={threshold})')
    axes[1].axis('off')
    
    if mask_path and Path(mask_path).exists():
        gt_mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        axes[2].imshow(gt_mask, cmap='gray')
        axes[2].set_title('Ground Truth')
        axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return pred_mask


## Part 8: Qualitative Evaluation — Predictions vs Ground Truth

Visual inspection of model predictions on the test set is critical in medical image segmentation. Quantitative metrics like F1 and accuracy can mask clinically relevant errors such as:
- Imprecise borders around glomeruli (false positives/negatives at edges)
- False positives in glomerulus-like structures (cracks, debris)
- Missed glomeruli in hard-to-segment regions

The cells below visualize predictions alongside ground truth for qualitative assessment.